In [ ]:
import os
import random
from pathlib import Path
from collections import Counter, defaultdict

# =========================================================
# CONFIG
# =========================================================
SEED = 42
SUBSET_FRACTION = 0.40

# =========================================================
# FIND REAL REPO ROOT
# =========================================================
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "models").exists() and (p / "configs").exists():
            return p
    raise RuntimeError(
        f"Could not find repo root from {start}. "
        "Expected a parent folder containing 'models/' and 'configs/'."
    )

REPO_ROOT = find_repo_root(Path.cwd())
print("Current working directory :", Path.cwd().resolve())
print("Detected repo root        :", REPO_ROOT)

DATASETS_DIR = REPO_ROOT / "datasets"
PROTOCOLS_DIR = DATASETS_DIR / "protocols"
UTMOS_PATH = DATASETS_DIR / "utmos" / "ASVspoof2019_train.txt"
REPORTS_DIR = REPO_ROOT / "reports" / "notes"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_TXT = REPORTS_DIR / "data_audit_results.txt"

SPLITS = {
    "ASVspoof2019_LA_train": DATASETS_DIR / "ASVspoof2019_LA_train" / "wav",
    "ASVspoof2019_LA_dev":   DATASETS_DIR / "ASVspoof2019_LA_dev"   / "wav",
    "ASVspoof2021_LA_eval":  DATASETS_DIR / "ASVspoof2021_LA_eval"  / "wav",
    "ASVspoof2021_DF_eval":  DATASETS_DIR / "ASVspoof2021_DF_eval"  / "wav",
}

EXPECTED_PROTOCOL_FILENAMES = {
    "ASVspoof2019_LA_train": "ASVspoof2019.LA.cm.train.trn.txt",
    "ASVspoof2019_LA_dev":   "ASVspoof2019.LA.cm.dev.trl.txt",
    "ASVspoof2021_LA_eval":  "ASVspoof2021.LA.cm.eval.trl.txt",
    "ASVspoof2021_DF_eval":  "ASVspoof2021.DF.cm.eval.trl.txt",
}

TRAIN_SUBSET_PROTOCOL = PROTOCOLS_DIR / "ASVspoof2019.LA.cm.train.trn.40pct.seed42.txt"
DEV_SUBSET_PROTOCOL   = PROTOCOLS_DIR / "ASVspoof2019.LA.cm.dev.trl.40pct.seed42.txt"

# =========================================================
# HELPERS
# =========================================================
def banner(msg):
    print("\n" + "=" * 100)
    print(msg)
    print("=" * 100)

def find_protocol_file(filename: str):
    # 1) exact expected location
    exact = PROTOCOLS_DIR / filename
    if exact.exists():
        return exact

    # 2) search anywhere inside repo
    matches = list(REPO_ROOT.rglob(filename))
    if matches:
        return matches[0]

    return None

def list_audio_ids(audio_dir: Path):
    """
    Returns:
      ids: set of utterance ids without extension
      count: total audio files
      ext_counter: counts by extension
    Supports both .wav and .flac
    """
    ids = set()
    ext_counter = Counter()

    if not audio_dir.exists():
        return ids, 0, ext_counter

    for ext in ("*.wav", "*.flac"):
        files = list(audio_dir.glob(ext))
        for f in files:
            ids.add(f.stem)
            ext_counter[f.suffix.lower()] += 1

    return ids, sum(ext_counter.values()), ext_counter

def read_nonempty_lines(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f if line.strip()]

def extract_utt_id(line: str):
    toks = line.strip().split()
    if not toks:
        return None

    if len(toks) == 1:
        return Path(toks[0]).stem

    for tok in toks:
        stem = Path(tok).stem
        if stem.startswith(("LA_", "DF_", "PA_")):
            return stem

    if len(toks) >= 2:
        return Path(toks[1]).stem

    return Path(toks[0]).stem

def extract_label(line: str):
    toks = line.strip().split()
    if not toks:
        return None
    last = toks[-1].lower()
    if last in {"bonafide", "spoof"}:
        return last
    return None

def protocol_crosscheck(split_name, protocol_path, audio_ids):
    lines = read_nonempty_lines(protocol_path)
    proto_ids = [extract_utt_id(line) for line in lines]
    proto_ids = [x for x in proto_ids if x is not None]

    proto_id_set = set(proto_ids)
    missing = sorted([utt for utt in proto_ids if utt not in audio_ids])
    extra = sorted([utt for utt in audio_ids if utt not in proto_id_set])

    return {
        "split": split_name,
        "protocol_file": str(protocol_path),
        "protocol_lines": len(lines),
        "protocol_ids": len(proto_ids),
        "audio_count": len(audio_ids),
        "missing_from_audio": missing,
        "extra_audio_not_in_protocol": extra,
    }

def parse_utmos_file(path: Path):
    if not path.exists():
        return None, None

    lines = read_nonempty_lines(path)
    ids = []
    bad_lines = []

    for i, line in enumerate(lines, start=1):
        toks = line.split()
        if len(toks) < 2:
            bad_lines.append((i, line))
            continue

        utt_id = Path(toks[0]).stem
        try:
            float(toks[1])
        except Exception:
            bad_lines.append((i, line))
            continue

        ids.append(utt_id)

    return {
        "line_count": len(lines),
        "valid_id_count": len(ids),
        "unique_id_count": len(set(ids)),
        "ids": set(ids),
        "bad_lines": bad_lines[:10],
    }, lines

def stratified_subset_from_protocol(lines, fraction=0.40, seed=42):
    by_label = defaultdict(list)
    for line in lines:
        label = extract_label(line)
        if label not in {"bonafide", "spoof"}:
            raise ValueError(f"Could not parse label from line:\n{line}")
        by_label[label].append(line)

    rng = random.Random(seed)
    subset_lines = []
    summary = {}

    for label in ["bonafide", "spoof"]:
        full_lines = by_label[label]
        n_full = len(full_lines)
        n_take = int(n_full * fraction)
        n_take = max(1, n_take) if n_full > 0 else 0

        selected = rng.sample(full_lines, n_take)
        subset_lines.extend(selected)

        summary[label] = {"full": n_full, "subset": n_take}

    rng.shuffle(subset_lines)

    summary["total_full"] = sum(v["full"] for v in summary.values() if isinstance(v, dict))
    summary["total_subset"] = sum(v["subset"] for v in summary.values() if isinstance(v, dict))
    return subset_lines, summary

def write_lines(path: Path, lines):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for line in lines:
            f.write(line + "\n")

# =========================================================
# RESOLVE PROTOCOL PATHS FIRST
# =========================================================
banner("Locate required protocol files")
PROTOCOL_FILES = {}
missing_protocol_names = []

for split, fname in EXPECTED_PROTOCOL_FILENAMES.items():
    p = find_protocol_file(fname)
    PROTOCOL_FILES[split] = p
    if p is None:
        missing_protocol_names.append(fname)
        print(f"[MISSING] {fname}")
    else:
        print(f"[FOUND] {fname} -> {p}")

# =========================================================
# CHECK 1 — Count files
# =========================================================
banner("CHECK 1 — Count audio files in each split (.wav + .flac)")
audio_id_map = {}
count_map = {}
ext_map = {}

for split, audio_dir in SPLITS.items():
    audio_ids, audio_count, ext_counter = list_audio_ids(audio_dir)
    audio_id_map[split] = audio_ids
    count_map[split] = audio_count
    ext_map[split] = dict(ext_counter)

    print(f"{split:22s} : {audio_count} files | {dict(ext_counter)} | dir={audio_dir}")

# =========================================================
# CHECK 2 — Protocol vs audio cross-check
# =========================================================
banner("CHECK 2 — Protocol vs audio cross-check")
crosschecks = {}

for split, proto_path in PROTOCOL_FILES.items():
    if proto_path is None:
        print(f"\n[{split}] SKIPPED — protocol file not found")
        crosschecks[split] = None
        continue

    result = protocol_crosscheck(split, proto_path, audio_id_map[split])
    crosschecks[split] = result

    print(f"\n[{split}]")
    print("Protocol file            :", result["protocol_file"])
    print("Protocol lines           :", result["protocol_lines"])
    print("Parsed utterance ids     :", result["protocol_ids"])
    print("Audio files              :", result["audio_count"])
    print("Missing protocol->audio  :", len(result["missing_from_audio"]))
    print("Extra audio not in proto :", len(result["extra_audio_not_in_protocol"]))

    if result["missing_from_audio"]:
        print("First 10 missing ids     :", result["missing_from_audio"][:10])

# =========================================================
# CHECK 3 — UTMOS coverage
# =========================================================
banner("CHECK 3 — UTMOS coverage")
utmos_info, utmos_raw_lines = parse_utmos_file(UTMOS_PATH)

if utmos_info is None:
    print(f"UTMOS file is MISSING: {UTMOS_PATH}")
    utmos_status = "MISSING"
    utmos_train_missing = None
else:
    train_ids = audio_id_map["ASVspoof2019_LA_train"]
    utmos_train_missing = sorted(list(train_ids - utmos_info["ids"]))

    print("UTMOS file              :", UTMOS_PATH)
    print("UTMOS total lines       :", utmos_info["line_count"])
    print("UTMOS valid ids         :", utmos_info["valid_id_count"])
    print("UTMOS unique ids        :", utmos_info["unique_id_count"])
    print("Train audio count       :", len(train_ids))
    print("Train ids missing MOS   :", len(utmos_train_missing))

    if utmos_info["bad_lines"]:
        print("Bad UTMOS lines         :", utmos_info["bad_lines"])
    if utmos_train_missing:
        print("First 10 missing MOS ids:", utmos_train_missing[:10])

    if utmos_info["line_count"] == len(train_ids) and len(utmos_train_missing) == 0:
        utmos_status = "PASS"
    else:
        utmos_status = "FAIL"

# =========================================================
# CHECK 4 — Class balance from training protocol
# =========================================================
banner("CHECK 4 — Class balance from training protocol")

train_proto = PROTOCOL_FILES["ASVspoof2019_LA_train"]
dev_proto   = PROTOCOL_FILES["ASVspoof2019_LA_dev"]

if train_proto is None:
    raise FileNotFoundError(
        "Training protocol file not found anywhere in repo. "
        "Please make sure ASVspoof2019.LA.cm.train.trn.txt exists."
    )
if dev_proto is None:
    raise FileNotFoundError(
        "Dev protocol file not found anywhere in repo. "
        "Please make sure ASVspoof2019.LA.cm.dev.trl.txt exists."
    )

train_lines = read_nonempty_lines(train_proto)
train_labels = [extract_label(line) for line in train_lines]

if any(lbl is None for lbl in train_labels):
    bad_idx = [i for i, lbl in enumerate(train_labels) if lbl is None][:10]
    raise ValueError(f"Could not parse label in train protocol lines at indices: {bad_idx}")

train_counter = Counter(train_labels)
n_bonafide = train_counter["bonafide"]
n_spoof = train_counter["spoof"]
imbalance_ratio = round(n_spoof / n_bonafide, 4) if n_bonafide > 0 else None

print("Train bonafide :", n_bonafide)
print("Train spoof    :", n_spoof)
print("Spoof/bonafide :", imbalance_ratio)

# =========================================================
# CREATE 40% STRATIFIED SUBSET (TRAIN + DEV ONLY)
# =========================================================
banner("CREATE 40% stratified subset (seed=42) for train/dev only")

dev_lines = read_nonempty_lines(dev_proto)

train_subset_lines, train_subset_summary = stratified_subset_from_protocol(
    train_lines, fraction=SUBSET_FRACTION, seed=SEED
)
dev_subset_lines, dev_subset_summary = stratified_subset_from_protocol(
    dev_lines, fraction=SUBSET_FRACTION, seed=SEED
)

write_lines(TRAIN_SUBSET_PROTOCOL, train_subset_lines)
write_lines(DEV_SUBSET_PROTOCOL, dev_subset_lines)

train_subset_ids = {extract_utt_id(line) for line in train_subset_lines}
dev_subset_ids = {extract_utt_id(line) for line in dev_subset_lines}

missing_train_subset_audio = sorted(list(train_subset_ids - audio_id_map["ASVspoof2019_LA_train"]))
missing_dev_subset_audio = sorted(list(dev_subset_ids - audio_id_map["ASVspoof2019_LA_dev"]))

print("Train subset protocol :", TRAIN_SUBSET_PROTOCOL)
print("Dev subset protocol   :", DEV_SUBSET_PROTOCOL)
print()
print("Train subset summary  :", train_subset_summary)
print("Dev subset summary    :", dev_subset_summary)
print()
print("Missing train subset audio ids :", len(missing_train_subset_audio))
print("Missing dev subset audio ids   :", len(missing_dev_subset_audio))

if missing_train_subset_audio:
    print("First 10 missing train ids:", missing_train_subset_audio[:10])
if missing_dev_subset_audio:
    print("First 10 missing dev ids  :", missing_dev_subset_audio[:10])

# =========================================================
# SAVE AUDIT REPORT
# =========================================================
banner("SAVE data_audit_results.txt")

lines_out = []
lines_out.append("NACL-SDD DATA AUDIT REPORT")
lines_out.append(f"Current working directory: {Path.cwd().resolve()}")
lines_out.append(f"Detected repo root: {REPO_ROOT}")
lines_out.append(f"Seed: {SEED}")
lines_out.append(f"Subset fraction: {SUBSET_FRACTION}")
lines_out.append("")

lines_out.append("1) FILE COUNTS")
for split in SPLITS:
    lines_out.append(f"{split}: {count_map[split]} | extensions={ext_map[split]}")
lines_out.append("")

lines_out.append("2) PROTOCOL vs AUDIO CROSS-CHECK")
for split, result in crosschecks.items():
    lines_out.append(f"[{split}]")
    if result is None:
        lines_out.append("protocol_file=NOT_FOUND")
        lines_out.append("")
        continue
    lines_out.append(f"protocol_file={result['protocol_file']}")
    lines_out.append(f"protocol_lines={result['protocol_lines']}")
    lines_out.append(f"protocol_ids={result['protocol_ids']}")
    lines_out.append(f"audio_count={result['audio_count']}")
    lines_out.append(f"missing_protocol_to_audio={len(result['missing_from_audio'])}")
    lines_out.append(f"extra_audio_not_in_protocol={len(result['extra_audio_not_in_protocol'])}")
    if result["missing_from_audio"]:
        lines_out.append(f"first_missing_ids={result['missing_from_audio'][:10]}")
    lines_out.append("")

lines_out.append("3) UTMOS COVERAGE")
lines_out.append(f"utmos_path={UTMOS_PATH}")
lines_out.append(f"utmos_status={utmos_status}")
if utmos_info is not None:
    lines_out.append(f"utmos_total_lines={utmos_info['line_count']}")
    lines_out.append(f"utmos_valid_ids={utmos_info['valid_id_count']}")
    lines_out.append(f"utmos_unique_ids={utmos_info['unique_id_count']}")
    lines_out.append(f"train_audio_count={len(audio_id_map['ASVspoof2019_LA_train'])}")
    lines_out.append(f"train_ids_missing_mos={len(utmos_train_missing)}")
    if utmos_train_missing:
        lines_out.append(f"first_missing_mos_ids={utmos_train_missing[:10]}")
else:
    lines_out.append("UTMOS file missing — CL and CL+DT cannot run until this is fixed.")
lines_out.append("")

lines_out.append("4) CLASS BALANCE")
lines_out.append(f"bonafide={n_bonafide}")
lines_out.append(f"spoof={n_spoof}")
lines_out.append(f"spoof_to_bonafide_ratio={imbalance_ratio}")
lines_out.append("")

lines_out.append("5) 40% STRATIFIED SUBSET CREATED (TRAIN/DEV ONLY)")
lines_out.append(f"train_subset_protocol={TRAIN_SUBSET_PROTOCOL}")
lines_out.append(f"dev_subset_protocol={DEV_SUBSET_PROTOCOL}")
lines_out.append(f"train_subset_bonafide={train_subset_summary['bonafide']['subset']}")
lines_out.append(f"train_subset_spoof={train_subset_summary['spoof']['subset']}")
lines_out.append(f"train_subset_total={train_subset_summary['total_subset']}")
lines_out.append(f"dev_subset_bonafide={dev_subset_summary['bonafide']['subset']}")
lines_out.append(f"dev_subset_spoof={dev_subset_summary['spoof']['subset']}")
lines_out.append(f"dev_subset_total={dev_subset_summary['total_subset']}")
lines_out.append(f"missing_train_subset_audio={len(missing_train_subset_audio)}")
lines_out.append(f"missing_dev_subset_audio={len(missing_dev_subset_audio)}")
lines_out.append("")
lines_out.append("NOTE: Evaluation sets remain FULL. Only train/dev were subsetted.")

with open(AUDIT_TXT, "w", encoding="utf-8") as f:
    f.write("\n".join(lines_out))

print(f"Saved audit report to: {AUDIT_TXT}")

# =========================================================
# FINAL SUMMARY
# =========================================================
banner("FINAL SUMMARY")
print(f"Train count        : {count_map['ASVspoof2019_LA_train']}")
print(f"Dev count          : {count_map['ASVspoof2019_LA_dev']}")
print(f"LA eval count      : {count_map['ASVspoof2021_LA_eval']}")
print(f"DF eval count      : {count_map['ASVspoof2021_DF_eval']}")
print()
print(f"Train bonafide     : {n_bonafide}")
print(f"Train spoof        : {n_spoof}")
print(f"Imbalance ratio    : {imbalance_ratio}")
print()
print(f"UTMOS status       : {utmos_status}")
print(f"Train subset total : {train_subset_summary['total_subset']}")
print(f"Dev subset total   : {dev_subset_summary['total_subset']}")
print()
print("Subset protocol files created:")
print(" -", TRAIN_SUBSET_PROTOCOL)
print(" -", DEV_SUBSET_PROTOCOL)
print()
print("Audit text saved to:")
print(" -", AUDIT_TXT)

Current working directory : /data/Sajjan_Singh/phd/nacl_sdd/notebooks
Detected repo root        : /data/Sajjan_Singh/phd/nacl_sdd

Locate required protocol files
[FOUND] ASVspoof2019.LA.cm.train.trn.txt -> /data/Sajjan_Singh/phd/nacl_sdd/_stage/asvspoof2019_la/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt
[FOUND] ASVspoof2019.LA.cm.dev.trl.txt -> /data/Sajjan_Singh/phd/nacl_sdd/_stage/asvspoof2019_la/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt
[MISSING] ASVspoof2021.LA.cm.eval.trl.txt
[FOUND] ASVspoof2021.DF.cm.eval.trl.txt -> /data/Sajjan_Singh/phd/nacl_sdd/datasets/ASVspoof2021_DF_eval/ASVspoof2021.DF.cm.eval.trl.txt

CHECK 1 — Count audio files in each split (.wav + .flac)
ASVspoof2019_LA_train  : 0 files | {} | dir=/data/Sajjan_Singh/phd/nacl_sdd/datasets/ASVspoof2019_LA_train/wav
ASVspoof2019_LA_dev    : 0 files | {} | dir=/data/Sajjan_Singh/phd/nacl_sdd/datasets/ASVspoof2019_LA_dev/wav
ASVspoof2021_LA_eval   : 0 files | {} | dir=/data/Sajjan_

In [1]:
# 01_data_audit.ipynb — ONE CELL ONLY (CORRECTED)
import random
from pathlib import Path
from collections import Counter, defaultdict
import re

SEED = 42
SUBSET_FRACTION = 0.40

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "models").exists() and (p / "configs").exists():
            return p
    raise RuntimeError("Could not find repo root.")

REPO_ROOT = find_repo_root(Path.cwd())
DATASETS_DIR = REPO_ROOT / "datasets"
PROTOCOLS_DIR = DATASETS_DIR / "protocols"
UTMOS_PATH = DATASETS_DIR / "utmos" / "ASVspoof2019_train.txt"
REPORTS_DIR = REPO_ROOT / "reports" / "notes"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_TXT = REPORTS_DIR / "data_audit_results.txt"

SPLITS = {
    "ASVspoof2019_LA_train": DATASETS_DIR / "ASVspoof2019_LA_train" / "wav",
    "ASVspoof2019_LA_dev":   DATASETS_DIR / "ASVspoof2019_LA_dev" / "wav",
    "ASVspoof2021_LA_eval":  DATASETS_DIR / "ASVspoof2021_LA_eval" / "wav",
    "ASVspoof2021_DF_eval":  DATASETS_DIR / "ASVspoof2021_DF_eval" / "wav",
}

PROTOCOL_FILES = {
    "ASVspoof2019_LA_train": PROTOCOLS_DIR / "ASVspoof2019.LA.cm.train.trn.txt",
    "ASVspoof2019_LA_dev":   PROTOCOLS_DIR / "ASVspoof2019.LA.cm.dev.trl.txt",
    "ASVspoof2021_LA_eval":  PROTOCOLS_DIR / "ASVspoof2021.LA.cm.eval.trl.txt",
    "ASVspoof2021_DF_eval":  PROTOCOLS_DIR / "ASVspoof2021.DF.cm.eval.trl.txt",
}

TRAIN_SUBSET_PROTOCOL = PROTOCOLS_DIR / "ASVspoof2019.LA.cm.train.trn.40pct.seed42.txt"
DEV_SUBSET_PROTOCOL   = PROTOCOLS_DIR / "ASVspoof2019.LA.cm.dev.trl.40pct.seed42.txt"

def banner(msg):
    print("\n" + "=" * 100)
    print(msg)
    print("=" * 100)

def list_audio_ids(audio_dir: Path):
    ids = set()
    ext_counter = Counter()
    if not audio_dir.exists():
        return ids, 0, ext_counter
    for ext in ("*.wav", "*.flac"):
        for f in audio_dir.glob(ext):
            ids.add(f.stem)
            ext_counter[f.suffix.lower()] += 1
    return ids, sum(ext_counter.values()), ext_counter

def read_nonempty_lines(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Protocol file not found: {path}")
    with open(path, "r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f if line.strip()]

def extract_utt_id(line: str):
    toks = line.strip().split()
    if not toks:
        return None

    # Prefer real utterance IDs, not speaker IDs
    for tok in toks:
        stem = Path(tok).stem
        if re.match(r"^(LA|DF|PA)_[TDE]_", stem):
            return stem

    # single-column eval protocol fallback
    if len(toks) == 1:
        return Path(toks[0]).stem

    # standard 2019 train/dev usually keep utt_id in second token
    return Path(toks[1]).stem

def extract_label(line: str):
    toks = line.strip().split()
    if not toks:
        return None
    last = toks[-1].lower()
    if last in {"bonafide", "spoof"}:
        return last
    return None

def protocol_crosscheck(split_name, protocol_path, audio_ids):
    lines = read_nonempty_lines(protocol_path)
    proto_ids = [extract_utt_id(line) for line in lines]
    proto_ids = [x for x in proto_ids if x is not None]
    proto_id_set = set(proto_ids)

    missing = sorted([utt for utt in proto_id_set if utt not in audio_ids])
    extra = sorted([utt for utt in audio_ids if utt not in proto_id_set])

    return {
        "split": split_name,
        "protocol_file": str(protocol_path),
        "protocol_lines": len(lines),
        "protocol_ids": len(proto_ids),
        "audio_count": len(audio_ids),
        "missing_from_audio": missing,
        "extra_audio_not_in_protocol": extra,
    }

def parse_utmos_file(path: Path):
    if not path.exists():
        return None
    lines = read_nonempty_lines(path)
    ids = []
    bad_lines = []
    for i, line in enumerate(lines, start=1):
        toks = line.split()
        if len(toks) < 2:
            bad_lines.append((i, line))
            continue
        utt_id = Path(toks[0]).stem
        try:
            float(toks[1])
        except Exception:
            bad_lines.append((i, line))
            continue
        ids.append(utt_id)
    return {
        "line_count": len(lines),
        "valid_id_count": len(ids),
        "unique_id_count": len(set(ids)),
        "ids": set(ids),
        "bad_lines": bad_lines[:10],
    }

def stratified_subset_from_protocol(lines, fraction=0.40, seed=42):
    by_label = defaultdict(list)
    for line in lines:
        label = extract_label(line)
        if label not in {"bonafide", "spoof"}:
            raise ValueError(f"Could not parse label from line:\n{line}")
        by_label[label].append(line)

    rng = random.Random(seed)
    subset_lines = []
    summary = {}

    for label in ["bonafide", "spoof"]:
        full_lines = by_label[label]
        n_full = len(full_lines)
        n_take = max(1, int(n_full * fraction)) if n_full > 0 else 0
        selected = rng.sample(full_lines, n_take)
        subset_lines.extend(selected)
        summary[label] = {"full": n_full, "subset": n_take}

    rng.shuffle(subset_lines)
    summary["total_full"] = sum(v["full"] for v in summary.values() if isinstance(v, dict))
    summary["total_subset"] = sum(v["subset"] for v in summary.values() if isinstance(v, dict))
    return subset_lines, summary

def write_lines(path: Path, lines):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for line in lines:
            f.write(line + "\n")

banner("CHECK 1 — Count audio files in each split")
audio_id_map, count_map = {}, {}
for split, audio_dir in SPLITS.items():
    audio_ids, audio_count, exts = list_audio_ids(audio_dir)
    audio_id_map[split] = audio_ids
    count_map[split] = audio_count
    print(f"{split:22s} : {audio_count} | {dict(exts)}")

banner("CHECK 2 — Protocol vs audio cross-check")
crosschecks = {}
for split, proto_path in PROTOCOL_FILES.items():
    if not proto_path.exists():
        print(f"[{split}] missing protocol -> {proto_path}")
        crosschecks[split] = None
        continue
    result = protocol_crosscheck(split, proto_path, audio_id_map[split])
    crosschecks[split] = result
    print(f"\n[{split}]")
    print("Protocol file            :", result["protocol_file"])
    print("Protocol lines           :", result["protocol_lines"])
    print("Parsed utterance ids     :", result["protocol_ids"])
    print("Audio files              :", result["audio_count"])
    print("Missing protocol->audio  :", len(result["missing_from_audio"]))
    print("Extra audio not in proto :", len(result["extra_audio_not_in_protocol"]))
    if result["missing_from_audio"]:
        print("First 10 missing ids     :", result["missing_from_audio"][:10])

banner("CHECK 3 — UTMOS coverage")
utmos_info = parse_utmos_file(UTMOS_PATH)
if utmos_info is None:
    print("UTMOS file missing:", UTMOS_PATH)
    utmos_status = "MISSING"
else:
    train_ids = audio_id_map["ASVspoof2019_LA_train"]
    utmos_train_missing = sorted(list(train_ids - utmos_info["ids"]))
    print("UTMOS total lines     :", utmos_info["line_count"])
    print("UTMOS valid ids       :", utmos_info["valid_id_count"])
    print("UTMOS unique ids      :", utmos_info["unique_id_count"])
    print("Train audio count     :", len(train_ids))
    print("Train ids missing MOS :", len(utmos_train_missing))
    utmos_status = "PASS" if len(utmos_train_missing) == 0 and utmos_info["line_count"] == len(train_ids) else "FAIL"

banner("CHECK 4 — Class balance from training protocol")
train_lines = read_nonempty_lines(PROTOCOL_FILES["ASVspoof2019_LA_train"])
dev_lines = read_nonempty_lines(PROTOCOL_FILES["ASVspoof2019_LA_dev"])

train_labels = [extract_label(x) for x in train_lines]
ctr = Counter(train_labels)
n_bonafide = ctr["bonafide"]
n_spoof = ctr["spoof"]
imbalance_ratio = round(n_spoof / n_bonafide, 4)
print("Train bonafide :", n_bonafide)
print("Train spoof    :", n_spoof)
print("Spoof/bonafide :", imbalance_ratio)

banner("CREATE 40% STRATIFIED SUBSET")
train_subset_lines, train_subset_summary = stratified_subset_from_protocol(train_lines, SUBSET_FRACTION, SEED)
dev_subset_lines, dev_subset_summary = stratified_subset_from_protocol(dev_lines, SUBSET_FRACTION, SEED)

write_lines(TRAIN_SUBSET_PROTOCOL, train_subset_lines)
write_lines(DEV_SUBSET_PROTOCOL, dev_subset_lines)

print("Train subset protocol :", TRAIN_SUBSET_PROTOCOL)
print("Dev subset protocol   :", DEV_SUBSET_PROTOCOL)
print("Train subset summary  :", train_subset_summary)
print("Dev subset summary    :", dev_subset_summary)

report_lines = []
report_lines.append("NACL-SDD DATA AUDIT REPORT")
report_lines.append(f"Detected repo root: {REPO_ROOT}")
report_lines.append(f"Seed: {SEED}")
report_lines.append(f"Subset fraction: {SUBSET_FRACTION}")
report_lines.append("")
for split in SPLITS:
    report_lines.append(f"{split}: {count_map[split]}")
report_lines.append("")
report_lines.append(f"bonafide={n_bonafide}")
report_lines.append(f"spoof={n_spoof}")
report_lines.append(f"spoof_to_bonafide_ratio={imbalance_ratio}")
report_lines.append(f"utmos_status={utmos_status}")
report_lines.append(f"train_subset_total={train_subset_summary['total_subset']}")
report_lines.append(f"dev_subset_total={dev_subset_summary['total_subset']}")

with open(AUDIT_TXT, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

banner("FINAL SUMMARY")
print("Audit saved to:", AUDIT_TXT)


CHECK 1 — Count audio files in each split
ASVspoof2019_LA_train  : 25380 | {'.wav': 25380}
ASVspoof2019_LA_dev    : 24986 | {'.wav': 24986}
ASVspoof2021_LA_eval   : 0 | {}
ASVspoof2021_DF_eval   : 88054 | {'.wav': 88054}

CHECK 2 — Protocol vs audio cross-check

[ASVspoof2019_LA_train]
Protocol file            : /data/Sajjan_Singh/phd/nacl_sdd/datasets/protocols/ASVspoof2019.LA.cm.train.trn.txt
Protocol lines           : 25380
Parsed utterance ids     : 25380
Audio files              : 25380
Missing protocol->audio  : 0
Extra audio not in proto : 0

[ASVspoof2019_LA_dev]
Protocol file            : /data/Sajjan_Singh/phd/nacl_sdd/datasets/protocols/ASVspoof2019.LA.cm.dev.trl.txt
Protocol lines           : 24844
Parsed utterance ids     : 24844
Audio files              : 24986
Missing protocol->audio  : 0
Extra audio not in proto : 142
[ASVspoof2021_LA_eval] missing protocol -> /data/Sajjan_Singh/phd/nacl_sdd/datasets/protocols/ASVspoof2021.LA.cm.eval.trl.txt

[ASVspoof2021_DF_eval]
Pro